# Sales & Profit Intelligence

**Portfolio case study · Synthetic data · Python / Pandas / Matplotlib**

This notebook calculates sales and profit metrics, checks data quality, compares product/category/region performance, and visualizes monthly trends. The dataset is synthetic and does not describe a real company. Calculated profit excludes operating expenses, taxes, and other business costs; it is not net profit.

Run from this notebook's folder. The shared metrics module resolves the CSV path relative to the project, independent of the kernel's working directory.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

current_directory = Path.cwd()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (current_directory, *current_directory.parents)
        if (candidate / "src" / "sales_profit_metrics.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate src/sales_profit_metrics.py from the current directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.sales_profit_metrics import (  # noqa: E402
    aggregate_metrics,
    format_percent,
    load_sales_data,
    summarize_metrics,
)

df = load_sales_data()
df.head()

## 2. Initial data-quality checks

Check shape, column types, missing values, duplicate rows, and invalid numeric values before calculating metrics.

In [ ]:
print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().to_frame('missing_values'))
print('Duplicate rows:', int(df.duplicated().sum()))

required_numeric = ['units_sold', 'unit_price', 'unit_cost', 'discount_rate']
invalid = (df[required_numeric].lt(0).any(axis=1) | df['discount_rate'].gt(1))
print('Rows with invalid numeric values:', int(invalid.sum()))
assert not df['order_date'].isna().any(), 'Missing order dates found'
assert not invalid.any(), 'Invalid numeric values found; investigate before analysis'

Revenue after discounts is gross sales multiplied by one minus the discount rate. Calculated profit is revenue after discounts less recorded product cost; it is not net profit. Monetary calculations retain decimal precision through aggregation, then round reported amounts to two decimals using ROUND_HALF_UP. Margins use unrounded aggregated totals and are formatted to one decimal percentage point.

In [ ]:
overall = summarize_metrics(df)
summary = pd.DataFrame({
    "metric": [
        "Orders/rows",
        "Units sold",
        "Revenue after discounts",
        "Recorded product cost",
        "Calculated profit",
        "Profit margin",
    ],
    "value": [
        overall["record_count"],
        overall["units_sold"],
        overall["revenue_after_discounts"],
        overall["recorded_product_cost"],
        overall["calculated_profit"],
        format_percent(overall["profit_margin"]),
    ],
})
display(summary)

## 4. Product, category, and region performance

In [ ]:
def performance_table(group_col):
    result = aggregate_metrics(df, group_col).set_index(group_col)
    result = result.rename(
        columns={
            "record_count": "records",
            "units_sold": "units",
            "revenue_after_discounts": "revenue",
            "recorded_product_cost": "cost",
            "calculated_profit": "profit",
        }
    )
    result["profit_margin"] = result["profit_margin"].map(format_percent)
    return result

display(performance_table("product"))
display(performance_table("category"))
display(performance_table("region"))

## 5. Visualize the results

These charts use calculated values from the dataset. Labels and chart titles state what is being measured.

In [ ]:
monthly = aggregate_metrics(df, "month").sort_values("month")
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly["month"], monthly["revenue_after_discounts"].map(float), marker="o", label="Revenue")
ax.plot(monthly["month"], monthly["calculated_profit"].map(float), marker="o", label="Calculated profit")
ax.set_title("Monthly Revenue and Calculated Profit (Synthetic Data)")
ax.set_xlabel("Month")
ax.set_ylabel("Amount (dataset currency units)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
by_product = performance_table("product").sort_values("profit", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(by_product.index, by_product["profit"].map(float))
ax.set_title("Calculated Profit by Product (Synthetic Data)")
ax.set_xlabel("Calculated profit (dataset currency units)")
ax.set_ylabel("Product")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## 6. Findings to investigate

The following cell prints the strongest observed groups. Treat these as findings from a synthetic demonstration dataset only, not as claims about a real business. Review the tables and charts before writing recommendations.

In [ ]:
category_results = performance_table("category")
region_results = performance_table("region")
product_results = performance_table("product")
overall = summarize_metrics(df)
print("Highest-profit category:", category_results["profit"].idxmax())
print("Highest-profit region:", region_results["profit"].idxmax())
print("Highest-profit product:", product_results["profit"].idxmax())
print("Overall profit margin:", format_percent(overall["profit_margin"]))

print("\nBusiness questions to investigate:")
print("- Do high-revenue products also have strong profit margins?")
print("- Which regions have opportunities to improve revenue or profitability?")
print("- How do discounts relate to profit across products?")
print("- Are monthly changes large enough to merit further investigation?")

## 7. Next steps

- Add automated tests for dashboard filter combinations and empty-result states.
- Compare discount scenarios without claiming causal effects.
- Recreate the core KPIs in Power BI.
- Replace the synthetic data only with a properly licensed public dataset or authorized business data.
- Update the report with verified results after running the notebook.